In [ ]:
import pandas as pd
from IPython.display import display

from study.data import cohort, prediction_metadata
from study.io import WORK, write_table
from study.statistics import paired_auc_interval, paired_difference, risk_matrix

In [ ]:
# Observation process summaries (R3) vs. physiological trajectories (R2)
predictions = pd.read_parquet(WORK / 'predictions/full')
rows = []

for (source, domain, support), group in predictions.groupby(['source', 'domain', 'support']):
    metadata = cohort(domain).sort_index()
    ids = (
        metadata.index[metadata.partition.eq('test')]
        if domain == 'eicu'
        else group.stay_id.drop_duplicates().sort_values()
    )

    # Align predictions from both tiers
    metadata = prediction_metadata(metadata.loc[ids])
    r3 = risk_matrix(group.loc[group.tier.eq('R3')], ids)
    r2 = risk_matrix(group.loc[group.tier.eq('R2')], ids)

    # Estimate paired differences with patient and hospital weighting
    for equal_hospital in [False, True] if domain == 'eicu' else [False]:
        result = paired_difference(metadata, r3, r2, cluster = domain == 'eicu', equal_hospital = equal_hospital)

        # Add AUROC intervals for the primary external comparisons
        if (
            source == 'metavision'
            and (not equal_hospital)
            and (domain == 'mimiciv' and support == 'shared' or (domain == 'eicu' and support == 'shared'))
        ):
            result.update(paired_auc_interval(metadata, r3, r2, domain == 'eicu', 2000, 77))

        rows.append(dict(source = source, domain = domain, support = support, contrast = 'R3_minus_R2', **result))

results = pd.DataFrame(rows)
write_table(WORK / 'results/observation.parquet', results)

display(results.round(5))